### Data Ingestion

In [5]:
### Document Structure

from langchain_core.documents import Document

In [6]:
doc=Document(
    page_content="This is the main content of the document. I am learning how to use RAG.",
    metadata={"source": "example.txt", "page": 1, "author": "Mohd Ahkam", "Date": "2024-06-01"}
)
doc

Document(metadata={'source': 'example.txt', 'page': 1, 'author': 'Mohd Ahkam', 'Date': '2024-06-01'}, page_content='This is the main content of the document. I am learning how to use RAG.')

In [7]:
## Creating a simple text file
import os 
os.makedirs("../data/text_files", exist_ok=True)

In [8]:
sample_text = {
    "../data/text_files/python_intro.txt":"""This is a sample text file for testing RAG. It contains multiple lines of text to simulate a real document. The purpose of this file is to provide content for retrieval and generation tasks in the RAG framework."""}
for file_path, content in sample_text.items():
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)
print("sample text file created successfully.")

sample text file created successfully.


In [9]:
### Text Loader
from langchain_community.document_loaders import TextLoader
loader = TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")
document = loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='This is a sample text file for testing RAG. It contains multiple lines of text to simulate a real document. The purpose of this file is to provide content for retrieval and generation tasks in the RAG framework.')]


In [10]:
## Directory Loader
from langchain_community.document_loaders import DirectoryLoader

## Load all text files in the directory
directory_loader = DirectoryLoader(
    "../data/text_files", glob="**/*.txt", ##pattern to match files
    loader_cls=TextLoader, ##Loader class to use
    loader_kwargs={'encoding':"utf-8"},
    show_progress=False
    
)
documents = directory_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='This is a sample text file for testing RAG. It contains multiple lines of text to simulate a real document. The purpose of this file is to provide content for retrieval and generation tasks in the RAG framework.')]

In [11]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, PyMuPDFLoader

## Load all pdf files in the directory
directory_loader = DirectoryLoader(
    "../data/pdf", glob="**/*.pdf", ##pattern to match files
    loader_cls=PyMuPDFLoader, ##Loader class to use
    show_progress=False
    
)
pdf_documents = directory_loader.load()
pdf_documents

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2024-08-12T20:56:23+05:30', 'source': '..\\data\\pdf\\Ahkam web Exp.pdf', 'file_path': '..\\data\\pdf\\Ahkam web Exp.pdf', 'total_pages': 1, 'format': 'PDF 1.7', 'title': '', 'author': 'aneesur rahman', 'subject': '', 'keywords': '', 'moddate': '2024-08-12T20:56:23+05:30', 'trapped': '', 'modDate': "D:20240812205623+05'30'", 'creationDate': "D:20240812205623+05'30'", 'page': 0}, page_content='TO WHOM SO EVER IT MAY CONCERN \n \n \nREF: VOL/EL/R2018023 \nDate: 31th July, 2024 \n \n \n \n \nThis is to certify that Mr. Mohd Ahkam was engaged as Associate with our organization as \nWeb Developer at Okhla, New Delhi from 05.07.2023 to 31.07.2024. \n \n \nHe was relieved from the engagement of the company on the above-mentioned date. \nHe had really shown good performance during his working period with our organization. \n \n \nWe wish good luck and success in all his future endeavors.

### Creating Data Chunks

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Parameters:
    - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks (preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [13]:
chunks = split_documents(documents)
chunks

Split 1 documents into 1 chunks

Example chunk:
Content: This is a sample text file for testing RAG. It contains multiple lines of text to simulate a real document. The purpose of this file is to provide content for retrieval and generation tasks in the RAG...
Metadata: {'source': '..\\data\\text_files\\python_intro.txt'}


[Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='This is a sample text file for testing RAG. It contains multiple lines of text to simulate a real document. The purpose of this file is to provide content for retrieval and generation tasks in the RAG framework.')]

### Embadding and VectorStoreDB

In [14]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict , Any , Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [15]:
class EmbeddingManager:
    """Handles document embedding using Sentence Transformers."""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        # all-MiniLM-L6-v2 is a popular model for sentence embeddings that balances performance and speed. and convert text to vectors for retrieval and generation tasks in RAG framework.
        """Initializing the embedding manager.
        Args:
            model_name: Hugging Face model name for sentence embeddings.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        """Loads the sentence transformer model."""
        try:
            print(f"Loading Embedding Model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model '{self.model_name}': {e}")
            raise
        
    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        """Generates an embedding for the given text.
        Args:
            text: The input text to embed.
        returns:
            A numpy array of embedding with shape (len(texts), embedding_dim).
        """
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embedding for text: {len(texts)} texts...")  # Print the characters of the text
        embedding = self.model.encode(texts, show_progress_bar=True)
        print(f"Embedding generated with shape: {embedding.shape}")
        return embedding
    
##Initializing the embedding manager and generating an embedding for a sample text
embedding_manager = EmbeddingManager()
embedding_manager

Loading Embedding Model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4541.72it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [16]:
class VectorStore:
    """Manages document embeddings in the vector store using ChromaDB."""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """Initialize the vector store.

        Args:
            collection_name: Name of the ChromaDB collection to use.
            persist_directory: Directory where the ChromaDB data will persist.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initializes the ChromaDB client and collection."""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Pdf document embeddings for RAG"},
            )
            print(f"Vector Store initialized collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing Vector Store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Adds documents to the vector store with their embeddings."""
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents and embeddings must match.")

        print(f"Adding {len(documents)} documents to the vector store...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata) if getattr(doc, "metadata", None) else {}
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist() if hasattr(embedding, "tolist") else list(embedding))

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text,
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection after addition: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to Vector Store: {e}")
            raise

    def add_document(self, documents: List[Any], embeddings: np.ndarray):
        """Backward-compatible wrapper."""
        return self.add_documents(documents, embeddings)


vector_store = VectorStore()
vector_store

Vector Store initialized collection: pdf_documents
Existing documents in collection: 2


In [17]:
chunks

[Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='This is a sample text file for testing RAG. It contains multiple lines of text to simulate a real document. The purpose of this file is to provide content for retrieval and generation tasks in the RAG framework.')]

In [18]:
texts = [chunk.page_content for chunk in chunks]
texts

['This is a sample text file for testing RAG. It contains multiple lines of text to simulate a real document. The purpose of this file is to provide content for retrieval and generation tasks in the RAG framework.']

In [19]:
### Convert Text to Embeddings
texts = [doc.page_content for doc in chunks]

### Generate embeddings
embeddings = embedding_manager.generate_embedding(texts)

### Store in vector store
vector_store.add_documents(chunks, embeddings)


Generating embedding for text: 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.11it/s]

Embedding generated with shape: (1, 384)
Adding 1 documents to the vector store...
Successfully added 1 documents to the vector store.
Total documents in collection after addition: 3


### Retriever Pipeline From VectorStore

In [20]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store."""
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initializes the retriever
        ARGS:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieves relevant documents for the query.
        Args:
            query: The input query string.
            top_k: Number of top results to return.
            score_threshold: Minimum cosine similarity score for results.
        Returns:
                A list of dictionaries containing retrieved documents and their metadata."""
        print(f"Retrieving documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold}")
        
        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embedding([query])[0]
        
        #Search in Vector Store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            # Process results
            retrieved_docs = []
            
            if results[document] and results[document][0]:  # Check if there are results
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance  # Convert distance to similarity score
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents retrieved from vector store.")
            return retrieved_docs
        
        
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
rag_retriever = RAGRetriever(vector_store, embedding_manager)

In [21]:
rag_retriever

In [22]:
rag_retriever.retrieve("What is RAG?")

Retrieving documents for query: 'What is RAG?' with top_k=5 and score_threshold=0.0
Generating embedding for text: 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 39.60it/s]

Embedding generated with shape: (1, 384)
Error during retrieval: cannot access local variable 'document' where it is not associated with a value


[]

### Integration Vectordb Context Pipeline With LLM Output

In [23]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()


### Initialize the Groq LLM
groq_api_key = "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"
llm = ChatGroq(groq_api_key=groq_api_key, model_name = "gemma2-9b-it", temperature=0.1, max_tokens=1024)

### Simple RAG function: Retriev contxt and generate answer
def rag_simple(query,retriever,llm, top_k=3):
    ## Retrieve the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant documents found. Generating answer without context."
    
    ## generate answer for Groq LLM
    prompt=f"""Use the following context to answer the question concisely.
        context: {context}
        question: {query}
        answer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [24]:
answer = rag_simple("who is certified?", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'who is certified?' with top_k=3 and score_threshold=0.0
Generating embedding for text: 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 74.00it/s]

Embedding generated with shape: (1, 384)
Error during retrieval: cannot access local variable 'document' where it is not associated with a value
No relevant documents found. Generating answer without context.


### Enhanced RAG Pipeline Features

In [25]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Hard Negative Mining Technqiues", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Hard Negative Mining Technqiues' with top_k=3 and score_threshold=0.1
Generating embedding for text: 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.71it/s]

Embedding generated with shape: (1, 384)
Error during retrieval: cannot access local variable 'document' where it is not associated with a value
Answer: No relevant context found.
Sources: []
Confidence: 0.0
Context Preview: 
